## Logistic Regression, Part 2: Regularization, Model Selection, and a Final Holdout Test

In this notebook, we are going to pull together a number of pieces that we've been discussing over the last several weeks. The goal of this is not only to explore logistic regression further, but also to pull in regularization and feature selection more explicitly, as well as explore what modeling looks like with the final holdout set at the end. Here is where we are going:

1. For simplicity, we will drop rows with missing data to start.
2. Set aside a **final holdout (test) set** (20%) that we do not touch until the very end.
3. Look at multicollinearity and drop multi-collinear variables
4. On the training set (80%), use **stratified k-fold cross-validation** to evaluate:
    * logistic regression with **no regularization**
    * **ridge** (L2) logistic regression for a range of `C` values
    * **lasso** (L1) logistic regression for a range of `C` values
5. For each of these models we will look at the usual metrics: log loss, confusion matrix, AUC
6. Once we have chosen the best model, we will refit it on **all** of the training data, and evaluate it **once** on the holdout set.

*AI disclosure: The code in this notebook was generated with the assistance of Claude. I gave Claude a pretty detailed prompt, then I went through and modified both the code itself and the explanations provided. I made sure I could execute and validate the output for code written.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import log_loss, roc_auc_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

### 1. Load the data and remove missing values

In [ ]:
fram = pd.read_csv("../data/framingham.csv")
print(fram.shape)
fram.isna().sum()

In [ ]:
# Drop any row with a missing value (about 14% of rows)
fram = fram.dropna()
print(fram.shape)

### 2. Train/test split (final holdout set)

We do a **single** 80/20 split. `stratify=y` keeps the ~15% CHD rate the same in both pieces. (I did not explicitly check CHD in this notebook because we have done it multiple times elsewhere.)

In [ ]:
y = fram["TenYearCHD"]
X = fram.drop(columns="TenYearCHD")   # Keep ALL features for now

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape}, CHD rate = {y_train.mean():.4f}")
print(f"Test set:     {X_test.shape}, CHD rate = {y_test.mean():.4f}")

### 3. Checking for multicollinearity and dropping variables if appropriate

Remember: If we only care about prediction accuracy, and we are doing regularization, we can leave multi-collinear variables in the model. But let's check just in case, via both a correlation plot and VIFs. 

In [ ]:
# Numeric columns in the dataset
nums = ["age", "cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]
corr = X_train[nums].corr()

sns.heatmap(
    corr, annot=True, fmt=".2f",
    cmap="RdBu", vmin=-1, vmax=1
)
plt.title("Correlation Heatmap (numeric variables)")
plt.show()

In [ ]:
X_vif = sm.add_constant(X_train[nums].dropna())

vifs = pd.Series([variance_inflation_factor(X_vif.values, i) for i in range(1, X_vif.shape[1])],
                 index=nums, name='VIF')
print(vifs.round(2))

### 4. Preprocessing, CV folds, and a baseline

Given the results above, we don't need to drop any of the numeric variables. 
For this model, we are going to use all of the numeric variables listed above, as well as male and currentSmoker, both of which are fully represented by a single column and do not need to be one hot encoded. 

**Note** I am again going to choose not to scale the categorical variables. Standard scaling will be used on the numeric variables, per usual. 

In [ ]:
bins = ["male", "currentSmoker"]
nums = ["age", "cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]

preprocess = ColumnTransformer(transformers=[
    ("bin", "passthrough", bins),
    ("num", StandardScaler(), nums),
])

# Use the same folds for every model, so the comparisons are fair
# Stratified KFold so that we preserve the percentage of CHD in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

**Here is how we will set the penalties.** 
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
* **No regularization:** `C=np.inf` (an infinitely weak penalty)
* **Ridge (L2):** `l1_ratio=0` (the default)
* **Lasso (L1):** `l1_ratio=1`, with a solver that supports it (`"saga"` or `"liblinear"`)

Remember that `C` is the **inverse** of the penalty strength: small `C` = strong penalty.

In [ ]:
# Let's define this helper function to build the model
def make_model(C=np.inf, l1_ratio=0):
    solver = "saga" if l1_ratio > 0 else "lbfgs"
    return Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(C=C, l1_ratio=l1_ratio, solver=solver, max_iter=1000)),
    ])

In [ ]:
# As discussed in class and explored in Lab Activity 6, we need a baseline log loss to compare to. 
# A useful model must beat this log loss.
# As before, we will predict 15% for each person
base_rate = y_train.mean()
ll_base = log_loss(y_train, np.full(len(y_train), base_rate))
print(f"Base-rate log loss (training set): {ll_base:.4f}")

**Helper Functions**

Per the instructions above, we are going to build several models, so we will create a few helper functions to help do this. 

* `cv_scores` uses `cross_validate` to get the log loss and AUC **on each fold**, and reports the mean and standard deviation across the 5 folds. 
* `cv_probs` uses `cross_val_predict` to get an **out-of-fold** predicted probability for every person in the training set. This means that each patient's probability comes from the model that did *not* see that patient. Pooling these gives us a **single** confusion matrix, ROC curve, and calibration plot instead of one per fold.  

Why not compute AUC from the pooled probabilities? The pooled probabilities come from 5 different fitted models, and AUC compares patients' probabilities against each other, including patients scored by different models. scikit-learn recommends computing AUC per fold, which is what cv_scores does. 

In [ ]:
# Note: cross_validate returns a numpy array, and calculates neg_log_loss and the AUC for each fold
# We are specifically interested in test_ values vs train_ values, which cross_validate also returns
# mean() and std() give mean and standard deviation across all 5 folds


def cv_scores(model):
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=["neg_log_loss", "roc_auc"])
    return {
        "log loss (mean)": -res["test_neg_log_loss"].mean(),
        "log loss (sd)": res["test_neg_log_loss"].std(),
        "AUC (mean)": res["test_roc_auc"].mean(),
        "AUC (sd)": res["test_roc_auc"].std(),
    }

def cv_probs(model):
    return cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

def precision_recall_table(y_true, p, thresholds=(0.5, 0.2)):
    rows = []
    for t in thresholds:
        y_hat = (p >= t).astype(int)
        rows.append({
            "threshold": t,
            "precision": precision_score(y_true, y_hat, zero_division=0),
            "recall": recall_score(y_true, y_hat),
        })
    return pd.DataFrame(rows)

def plot_confusion(y_true, p, title, thresholds=(0.5, 0.2)):
    fig, axes = plt.subplots(1, len(thresholds), figsize=(5 * len(thresholds), 4))
    for ax, t in zip(axes, thresholds):
        ConfusionMatrixDisplay.from_predictions(y_true, (p >= t).astype(int), ax=ax,
                                                cmap="Blues", colorbar=False)
        ax.set_title(f"{title}\nthreshold = {t}")
    plt.tight_layout()
    plt.show()

### 4a + 5. No regularization

### Log loss and AUC (5-fold CV on the training set)

In [ ]:
model_none = make_model(C=np.inf)

scores_none = cv_scores(model_none)
pd.Series(scores_none).round(4)

### Confusion matrix (pooled out-of-fold predictions)

In [ ]:
p_none = cv_probs(model_none)

# Log loss of the pooled out-of-fold probabilities: essentially the same as the per-fold mean above
print(f"Pooled out-of-fold log loss: {log_loss(y_train, p_none):.4f}")

plot_confusion(y_train, p_none, "No regularization (CV)")

precision_recall_table(y_train, p_none)

As in our previous in-lcass example, at a threshold of 0.5 the model only predicts that 57 people will get CHD, 35 of them correctly, with a high number of false negatives. Remember that most predicted probabilities are well below 0.5 because only 15% of patients have CHD. 

Lowering the threshold to 0.2 trades false negatives for false positives. At a threshold of 0.5 the precision (How many of the predicted positives are actually positive?) is around 61% but the recall (How many of the actual positives did we correctly predict?) is really low. If we lower the threshold, we do a better job predicting actual positives (recall goes up) but we also increase the number of false positives as a tradeoff, and lots of our predicted 1s aren't actually 1s. 

### ROC curve (pooled out-of-fold predictions)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_train, p_none, name="No regularization", ax=ax, plot_chance_level=True)
ax.set_title("ROC curve: 5-fold CV (pooled out-of-fold)")
plt.show()

### Thresholds on the ROC curve

Every threshold gives one confusion matrix, and so one (FPR, TPR) point. The ROC curve traces those points for **every** possible threshold. Below we mark a few specific thresholds, including the 0.5 and 0.2 used in the confusion matrices above to visualize this in a slightly different way. 

In [ ]:
thresholds = [0.05, 0.1, 0.2, 0.3, 0.5]

points = []
for t in thresholds:
    tn, fp, fn, tp = confusion_matrix(y_train, (p_none >= t).astype(int)).ravel()
    points.append({
        "threshold": t,
        "FPR": fp / (fp + tn),
        "TPR": tp / (tp + fn),
        "recall": tp / (tp + fn),
        "precision": tp / (tp + fp) if (tp + fp) > 0 else 0.0,
    })
points = pd.DataFrame(points)

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_train, p_none, name="No regularization", ax=ax, plot_chance_level=True)
ax.scatter(points["FPR"], points["TPR"], color="red", zorder=3, label="Thresholds")
for _, row in points.iterrows():
    ax.annotate(f"t = {row['threshold']}", (row["FPR"], row["TPR"]),
                textcoords="offset points", xytext=(8, -12))
ax.legend(loc="lower right")
ax.set_title("ROC curve with thresholds: 5-fold CV (pooled out-of-fold)")
plt.show()

points.round(3)

Comments on the plot above:
* **Lowering the threshold moves us up and to the right along the curve.** We predict CHD for more patients, so we catch more true cases (higher TPR) but also flag more patients who don't develop CHD (higher FPR).
* **t = 0.5** is near the bottom-left corner: almost no false positives (FPR ≈ 0.01), but we catch only about 8% of CHD cases.
* **t = 0.2** catches about half of the CHD cases (TPR ≈ 0.52) with a false positive rate of about 0.22.
* **t = 0.05** catches almost every case (TPR ≈ 0.96), but flags 80% of patients who don't develop CHD.
* The AUC summarizes the whole curve, over every threshold. That is why changing the threshold changes the confusion matrix but not the AUC.

### Calibration plot

A **calibration plot** asks a different question than AUC: when the model predicts a probability of 0.3, do about 30% of those patients actually develop CHD?

To make the plot, we sort patients into bins by their predicted probability. For each bin, we plot the **mean predicted probability** (x-axis) against the **fraction of patients who actually had CHD** (y-axis). A perfectly calibrated model falls on the diagonal.
* Points **below** the diagonal: the model's probabilities are too high (overconfident about CHD).
* Points **above** the diagonal: the model's probabilities are too low.

We use `strategy="quantile"` so that every bin has the same number of patients. With only 15% positives, most predicted probabilities are small, and equal-width bins would leave the high-probability bins nearly empty.

Calibration and AUC measure different things. 
* AUC asks if the model can rank positive cases above negative cases. 
* Calibration cares whether the probabilities themselves are accurate? Does p=0.3 mean that about 30% of those patients get CHD? 

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
CalibrationDisplay.from_predictions(y_train, p_none, n_bins=10, strategy="quantile",
                                    name="No regularization", ax=ax)
ax.set_title("Calibration plot: 5-fold CV (pooled out-of-fold)")
plt.show()

Comments on the calibration plot:
* The points lie close to the diagonal, so the model's probabilities are roughly accurate: patients with a predicted probability of about 0.2 developed CHD about 20% of the time.
* Almost all of the predicted probabilities are below 0.5. The highest bin has a mean predicted probability of about 0.43, so there are no points on the right half of the plot. This is the same reason the 0.5 threshold almost never predicts CHD.
* In the highest bin the model is a little overconfident: it predicts about 0.43 on average, but only about 38% of those patients had CHD.
* Logistic regression tends to be well calibrated because it is fit by minimizing log loss, which penalizes probabilities that are too high or too low. This may not be true for other types of models. 

### 4b+c +5 Ridge and lasso for different values of `C`

Now we repeat the cross-validation for ridge and lasso across a grid of `C` values. 

In [ ]:
C_values = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100]
# Note: We could also use GridSearchCV here! 

rows = []
for penalty, l1_ratio in [("ridge", 0), ("lasso", 1)]:
    for C in C_values:
        rows.append({"penalty": penalty, "C": C, **cv_scores(make_model(C=C, l1_ratio=l1_ratio))})

reg_results = pd.DataFrame(rows)
reg_results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for penalty, df in reg_results.groupby("penalty"):
    ax.errorbar(df["C"], df["log loss (mean)"], yerr=df["log loss (sd)"],
                marker="o", capsize=3, label=penalty)
ax.axhline(scores_none["log loss (mean)"], color="black", ls="--", label="no regularization")
ax.axhline(ll_base, color="red", ls=":", label="base rate (no predictors)")
ax.set(xscale="log", xlabel="C (log scale; smaller = stronger penalty)",
       ylabel="mean CV log loss (± 1 SD across folds)", title="CV log loss vs. C")
ax.legend()
plt.show()

Comments on the plot and table:
* **Large `C` (weak penalty):** both ridge and lasso converge to the no-regularization model (black dashed line), as expected.
* **Small `C` (strong penalty):** the log loss rises toward the base rate. Ridge shrinks the coefficients toward zero but never exactly to zero. Lasso sets **every** coefficient exactly to zero at `C` ≤ 0.003, so the model predicts the same probability for every patient. That is why its AUC is exactly 0.5 there. (These points sit at or slightly above the red line because the red line is computed on the full training set, while in CV each fold's CHD rate is used to predict a different fold.)
* **In between:** a small dip. According to the table above, the lowest log loss for ridge is 0.3799 vs 0.3802 with no regularization. This dip is tiny. The improvement (about 0.0003) is much smaller than the fold-to-fold standard deviation (about 0.008).

In [ ]:
# Best C for each penalty: the lowest mean CV log loss
best = reg_results.loc[reg_results.groupby("penalty")["log loss (mean)"].idxmin()].set_index("penalty")
C_ridge, C_lasso = best.loc["ridge", "C"], best.loc["lasso", "C"]
print(f"Best ridge C: {C_ridge}")
print(f"Best lasso C: {C_lasso}")

model_ridge = make_model(C=C_ridge, l1_ratio=0)
model_lasso = make_model(C=C_lasso, l1_ratio=1)

### Confusion matrices for the best ridge and lasso models

These are the same plots as in Section 4, using pooled out-of-fold predictions.

In [ ]:
p_ridge = cv_probs(model_ridge)
p_lasso = cv_probs(model_lasso)

plot_confusion(y_train, p_ridge, f"Ridge, C = {C_ridge} (CV)")
plot_confusion(y_train, p_lasso, f"Lasso, C = {C_lasso} (CV)")

In [ ]:
# Precision and recall for ridge
precision_recall_table(y_train, p_ridge)

In [ ]:
# Precision and recall for lasso
precision_recall_table(y_train, p_lasso)

### ROC curves and calibration plots for all three models

In [ ]:
probs = {
    "No regularization": p_none,
    f"Ridge (C = {C_ridge})": p_ridge,
    f"Lasso (C = {C_lasso})": p_lasso,
}

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for i, (name, p) in enumerate(probs.items()):
    RocCurveDisplay.from_predictions(y_train, p, name=name, ax=axes[0],
                                     plot_chance_level=(i == 0))
    CalibrationDisplay.from_predictions(y_train, p, n_bins=10, strategy="quantile",
                                        name=name, ax=axes[1])
axes[0].set_title("ROC curves: 5-fold CV (pooled out-of-fold)")
axes[1].set_title("Calibration: 5-fold CV (pooled out-of-fold)")
plt.tight_layout()
plt.show()

### What did the penalty do to the coefficients?

To see the coefficients, we fit each of the three models on the full training set. (We are only *looking* at the coefficients here; the model choice is based on the CV scores above.)

In [ ]:
coefs = {}
for name, model in [("none", model_none), ("ridge", model_ridge), ("lasso", model_lasso)]:
    model.fit(X_train, y_train)
    names = model.named_steps["preprocess"].get_feature_names_out()
    coefs[name] = pd.Series(model.named_steps["model"].coef_[0], index=names)

coef_table = pd.DataFrame(coefs)
coef_table["lasso_odds_ratio"] = np.exp(coef_table["lasso"])
coef_table.sort_values("none", key=abs, ascending=False).round(3)

Comments on the coefficients:
* **Strongest predictors are the same in all three models predictors.** `age`, `male`, `sysBP`, `cigsPerDay`, and `glucose` have the highest coefficients in all three models.
* **Ridge shrinks every coefficient vs no regularization, but not by very much.** Their log losses are also very similar.
* `male` shrinks the most. This is partially because we did not scale this variable.
* **Lasso sets 2 coefficients exactly to zero**: `currentSmoker` and `diaBP`. Both of these coefficients provide redundant information. 
    * Every non-smoker (`currentSmoker=0`) has `cigsPerDay=0` so once `cigsPerDay` is in the model, `currentSmoker` doesn't add much.
    *  `diaBP` is correlated with `sysBP` and so even though our VIF caluclation did not indicate we need to drop this variable, we know they are correlated.


Example interpretation:
* Age: Holding all other variables constant, each one standard deviation increase in age multiplies the odds of getting CHD in 10-years by 1.71 which is a 71% increase in the odds. (Note: One standard deviation because we used the standard scaler.)
* Male: Holding other variables constant, mean have 1.44 times the odds of CHD than women. (We did not scale, so the interpretation here is more straightforward.)

Caveat:

* **The penalty depends on scale.** We standardized the numeric variables but left the 0/1 variables unscaled. If we had standardized the 0/1 variables too, the penalty would treat them differently. 

## 7. Which model is best?

In [ ]:
summary = pd.DataFrame({
    "No regularization": scores_none,
    f"Ridge (C = {C_ridge})": cv_scores(model_ridge),
    f"Lasso (C = {C_lasso})": cv_scores(model_lasso),
}).T
summary.round(4)

By our criterion (lowest mean CV log loss), the best model is **lasso with `C = 0.1`**. It also has the highest mean AUC.

However, the three models are **essentially tied**:
* The differences in log loss and AUC are much smaller than the fold-to-fold standard deviations.
* The ROC curves and calibration curves above lie almost on top of each other.

Regularization doesn't help much here because we have about 2,900 training patients and only 10 coefficients, so the unpenalized model has little room to overfit. Regularization matters most when there are many predictors relative to the number of observations.

There is still a good reason to prefer the lasso model: it does just as well with 8 predictors instead of 10. When models perform about equally, the simpler one is easier to explain and use. 

## 8. Refit on all training data and test on the holdout set

The final model is fit on **all** of the training data (no more CV) and evaluated **once** on the test set. We do not go back and change the model based on what we see here; if we did, the test set would no longer be an honest estimate of performance on new data.

In [ ]:
# Our chosen model from Section 7: lasso with the best C from CV
final_model = make_model(C=C_lasso, l1_ratio=1)
final_model.fit(X_train, y_train)

p_test = final_model.predict_proba(X_test)[:, 1]

# Baseline for the test set: predict the TRAINING CHD rate for every test patient
ll_base_test = log_loss(y_test, np.full(len(y_test), base_rate))

print(f"Test log loss:             {log_loss(y_test, p_test):.4f}")
print(f"Test base-rate log loss:   {ll_base_test:.4f}")
print(f"Test AUC:                  {roc_auc_score(y_test, p_test):.4f}")
print()
print(f"CV log loss (training):    {summary.iloc[2]['log loss (mean)']:.4f} ± {summary.iloc[2]['log loss (sd)']:.4f}")
print(f"CV AUC (training):         {summary.iloc[2]['AUC (mean)']:.4f} ± {summary.iloc[2]['AUC (sd)']:.4f}")

In [ ]:
plot_confusion(y_test, p_test, f"Final lasso model (C = {C_lasso}), test set")
precision_recall_table(y_test, p_test)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
RocCurveDisplay.from_predictions(y_test, p_test, name="Final model", ax=axes[0], plot_chance_level=True)
CalibrationDisplay.from_predictions(y_test, p_test, n_bins=10, strategy="quantile",
                                    name="Final model", ax=axes[1])
axes[0].set_title("ROC curve: test set")
axes[1].set_title("Calibration: test set")
plt.tight_layout()
plt.show()

Comments on the holdout results:
* The test log loss (0.385) is a little worse than the CV estimate (0.379), and the test AUC (0.723) is a little lower than the CV AUC (0.736). Both are well within one fold-to-fold SD of the CV numbers.
* We expect the test score to be slightly worse. We used the CV scores to **choose** `C` and the model, so the winning CV score is a little optimistic. The test set was never used for any decision, so it gives the honest estimate. Also, the test set is small (732 patients, 111 with CHD), so the test numbers are themselves noisy.
* The model clearly beats the base rate on new data (0.385 vs. 0.426).
* **Confusion matrices:** at a threshold of 0.5, the model catches only 13 of the 111 CHD cases. At 0.2, it catches 57 of 111 (about half), at the cost of 127 false positives. Which threshold is better depends on the relative cost of a missed CHD case vs. a false alarm. That decision should be made before looking at the test set.
* **Calibration:** the test-set calibration plot is noisier than the CV plot (each bin has only about 73 patients and about 11 CHD cases), but the points still follow the diagonal for most of the range. As in CV, the highest bin is overconfident: a mean predicted probability of about 0.45 vs. about 35% observed.